# DSAA 5020 Final Project — Topic 8
## Can Transfer Learning Outperform Training from Scratch with Limited Labeled Data?

**Dataset**: Oxford-IIIT Pet (37 fine-grained pet breeds: 12 cat breeds, 25 dog breeds)  
**Primary Backbone**: `ResNet-18` (ImageNet-1K pretrained vs. random init)  
**Optional Cross-Backbone Check**: `MobileNetV3-Small`

### Before Running
1. In Colab menu, select **Runtime → Change runtime type → T4 GPU** (or L4/A100).
2. Upload this project folder (`src/`, `requirements.txt`, `colab_transfer_learning_pet.ipynb`) to Google Drive (e.g. `MyDrive/dsaa5020_topic8`).
3. Run cells top-to-bottom. All results are saved persistently to Google Drive and **automatically resume** if Colab disconnects.

In [ ]:
# 1. Check GPU & PyTorch environment
!nvidia-smi
import torch, torchvision
print(f"PyTorch: {torch.__version__}, TorchVision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Mount Google Drive and locate project directory
import os
from pathlib import Path

USE_GOOGLE_DRIVE = True  # Set False if running on a local GPU server
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/dsaa5020_topic8"  # Adjust if you named the folder differently

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
    # Keep raw image dataset on Colab local SSD for fast I/O, while persisting results to Google Drive
    DATA_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path.cwd()
    DATA_DIR = PROJECT_DIR / "data"

assert (PROJECT_DIR / "src").exists(), (
    f"Could not find src/ in {PROJECT_DIR}. "
    "Please upload the project folder to Google Drive and update DRIVE_PROJECT_DIR."
)
os.chdir(PROJECT_DIR)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working directory : {Path.cwd()}")
print(f"Dataset cache dir : {DATA_DIR}")
print(f"Results output dir: {RESULTS_DIR}")

In [ ]:
# 3. Install any missing packages (Colab already has torch/torchvision/sklearn/seaborn)
!pip install -q tqdm

### Step A: Run Experiments (`core`, `extended`, or `full_with_mobilenet`)

| Mode | Included Experiments | Approx. T4 GPU Runtime |
| :--- | :--- | :---: |
| `core` | 4 required conditions × 3 seeds = 12 runs | ~6–8 min |
| `extended` **(Recommended)** | Core + full 2×4 matrix (10% & 100%, 3 seeds) + Data Efficiency Curve (5%, 25%, 50%) = 36 runs | ~18–25 min |
| `full_with_mobilenet` | Everything in `extended` + MobileNetV3-Small cross-backbone check (+8 runs) | ~22–30 min |

> **Auto-Resume**: Completed runs are cached in `results/<backbone>/<run>/metrics.json`. Re-running this cell skips finished runs automatically.

In [ ]:
# Choose mode: "extended" (default) or "full_with_mobilenet" (adds 2nd lightweight backbone)
MODE = "full_with_mobilenet"

!python -m src.run_experiments \
    --mode {MODE} \
    --data_dir "{DATA_DIR}" \
    --output_dir "{RESULTS_DIR}" \
    --seeds 42 123 2026 \
    --curve_seeds 42 \
    --num_workers 2

### Step B: Generate All Tables & Publication-Grade Figures (Requirements 1–5)

In [ ]:
!python -m src.visualize --output_dir "{RESULTS_DIR}" --data_dir "{DATA_DIR}"

In [ ]:
# Preview Markdown Summary Table & all generated figures inline
from IPython.display import Markdown, Image, display

md_path = RESULTS_DIR / "tables" / "summary_table.md"
if md_path.exists():
    display(Markdown(md_path.read_text(encoding="utf-8")))

fig_dir = RESULTS_DIR / "figures"
for img_path in sorted(fig_dir.glob("*.png")):
    display(Markdown(f"#### {img_path.name}"))
    display(Image(filename=str(img_path), width=900))

In [ ]:
# Optional: Zip all tables and figures for one-click download to your laptop
import shutil

zip_base = Path("/content/topic8_results_bundle")
bundle_tmp = Path("/content/_bundle_tmp")
if bundle_tmp.exists():
    shutil.rmtree(bundle_tmp)
shutil.copytree(RESULTS_DIR / "figures", bundle_tmp / "figures")
shutil.copytree(RESULTS_DIR / "tables", bundle_tmp / "tables")
if (RESULTS_DIR / "eda_summary.json").exists():
    shutil.copy2(RESULTS_DIR / "eda_summary.json", bundle_tmp / "eda_summary.json")

zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=bundle_tmp)
print(f"Created bundle: {zip_path}")

if USE_GOOGLE_DRIVE:
    from google.colab import files
    files.download(zip_path)